# Test II — VLM-Based OCR Pipeline for Handwritten Early Modern Spanish Documents

The task requires the VLM to be used at **all stages** of the pipeline, not just as a late-stage cleanup step. This notebook:

1. Prepares the data (PDF → page images → line crops + ground truth alignment)
2. Fine-tunes Qwen2.5-VL-7B with LoRA on ground truth line-image pairs
3. Runs the 4-stage pipeline (layout → transcription → self-correction → contextual refinement)
4. Evaluates with CER, WER, NLS, and orthographic preservation metrics
5. Per-stage ablation showing where VLM integration provides the most gains
6. Compares fine-tuned model vs. zero-shot baseline

In [ ]:
# clone repo and install deps
!git clone https://github.com/invi-bhagyesh/ocr.git 2>/dev/null || echo "already cloned"
%cd ocr
!pip install -q -r requirements.txt
!apt-get install -y -q poppler-utils 2>/dev/null || echo "poppler already installed or not on apt"

In [ ]:
import os
from pathlib import Path

# create data dirs
for d in ["data/raw_pdfs", "data/pages", "data/lines", "data/ground_truth", "checkpoints"]:
    Path(d).mkdir(parents=True, exist_ok=True)

# set API key for Gemini (used for line detection baseline)
# os.environ["GEMINI_API_KEY"] = "your-key-here"

# data must be downloaded manually from SharePoint (requires browser auth):
#   PDFs:          https://bama365-my.sharepoint.com/.../Test%20sources
#   Transcriptions: https://bama365-my.sharepoint.com/.../Test%20transcriptions
#
# upload to cloud GPU, then:
# !unzip -q /path/to/sources.zip -d data/raw_pdfs/
# !unzip -q /path/to/transcriptions.zip -d data/ground_truth/

pdfs = list(Path("data/raw_pdfs").glob("*.pdf"))
gts = list(Path("data/ground_truth").iterdir())
print(f"PDF files: {len(pdfs)}")
for p in pdfs: print(f"  {p.name}")
print(f"GT files: {len(gts)}")
for g in gts: print(f"  {g.name}")

assert len(pdfs) > 0, "No PDFs found -- upload handwritten source PDFs to data/raw_pdfs/"
assert len(gts) > 0, "No ground truth found -- upload transcription files to data/ground_truth/"

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.data.pdf_convert import convert_all
from src.data.ground_truth import load_all_transcriptions, build_line_pairs
from src.vlm.client import get_client
from src.vlm.finetune import prepare_lora_model, train_lora, OCRFineTuneDataset
from src.pipeline.runner import OCRPipeline
from src.pipeline.stage1_layout import detect_lines, crop_lines
from src.eval.metrics import evaluate_lines, evaluate_ablation, count_normalizations
from src.utils.image import preprocess_page

PDF_DIR = "data/raw_pdfs"
PAGES_DIR = "data/pages"
GT_DIR = "data/ground_truth"
LINES_DIR = "data/lines"
CKPT_DIR = "checkpoints"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Data Preparation

Convert handwritten PDFs to page images, load ground truth, and create line-level training pairs.

In [ ]:
print("Converting PDFs to page images...")
all_pages = convert_all(PDF_DIR, PAGES_DIR, dpi=300)

print(f"\nSources: {len(all_pages)}")
for name, pages in all_pages.items():
    print(f"  {name}: {len(pages)} pages")

print("\nLoading ground truth transcriptions...")
transcriptions = load_all_transcriptions(GT_DIR)

In [ ]:
# display sample pages
fig, axes = plt.subplots(1, min(3, len(all_pages)), figsize=(18, 10))
if not hasattr(axes, '__len__'):
    axes = [axes]
for ax, (name, pages) in zip(axes, list(all_pages.items())[:3]):
    ax.imshow(Image.open(pages[0]))
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Pages from Handwritten Sources")
plt.tight_layout()
plt.show()

## 2. Line Segmentation and Training Data

Use hybrid line detection (horizontal projection profile + VLM filtering) to extract line images, then pair with ground truth for fine-tuning.

In [ ]:
# use a zero-shot client for initial line detection
baseline_client = get_client("gemini")

all_train_pairs = []
all_eval_data = {}

for source_name in all_pages:
    if source_name not in transcriptions:
        continue

    gt_lines = transcriptions[source_name]
    page_path = all_pages[source_name][0]
    line_dir = Path(LINES_DIR) / source_name

    print(f"\n{source_name}: detecting lines...")
    bboxes = detect_lines(page_path, baseline_client, method="hybrid")
    line_paths = crop_lines(page_path, bboxes, line_dir)
    print(f"  {len(line_paths)} lines detected, {len(gt_lines)} GT lines available")

    # pair GT lines with detected line images
    pairs = build_line_pairs(gt_lines, line_paths)
    print(f"  {len(pairs)} line-image/text pairs created")

    # split: first 80% for training, rest for eval
    split_idx = int(len(pairs) * 0.8)
    all_train_pairs.extend(pairs[:split_idx])
    all_eval_data[source_name] = {
        "pairs": pairs[split_idx:],
        "gt_lines": gt_lines,
        "line_paths": line_paths,
        "bboxes": bboxes,
    }

print(f"\nTotal training pairs: {len(all_train_pairs)}")
print(f"Eval sources: {list(all_eval_data.keys())}")

In [ ]:
# show sample line crops with their GT text
fig, axes = plt.subplots(min(5, len(all_train_pairs)), 1, figsize=(14, 8))
if not hasattr(axes, '__len__'):
    axes = [axes]
for ax, pair in zip(axes, all_train_pairs[:5]):
    ax.imshow(Image.open(pair["image_path"]))
    ax.set_title(f'GT: "{pair["text"][:80]}"', fontsize=9, loc="left")
    ax.axis("off")
plt.suptitle("Training Line-Image / Text Pairs")
plt.tight_layout()
plt.show()

## 3. Fine-Tune Qwen2.5-VL with LoRA

Fine-tune the VLM on our ground truth line-image pairs. This is the core of the task — adapting the model to early modern Spanish handwriting while preserving historical orthography.

LoRA config: r=16, alpha=32, 4-bit quantization. Targeting q/k/v/o projection layers.
Image augmentation: rotation ±2°, brightness/contrast jitter, Gaussian blur.

In [ ]:
from transformers import AutoProcessor

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = prepare_lora_model(MODEL_NAME, r=16, alpha=32)

adapter_path, ft_history = train_lora(
    model, processor, all_train_pairs,
    output_dir=CKPT_DIR, epochs=5, lr=2e-4,
    batch_size=2, grad_accum=8
)
print(f"\nAdapter saved to: {adapter_path}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ft_history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LoRA Fine-Tuning Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Run the 4-Stage Pipeline

Load the fine-tuned model and run the full pipeline with ablation tracking.

In [ ]:
# load fine-tuned model
finetuned_client = get_client("qwen", adapter_path=str(adapter_path))

# prepare few-shot examples (image + text pairs from training data)
few_shot = all_train_pairs[:3]

pipeline_ft = OCRPipeline(finetuned_client, few_shot_examples=few_shot, lines_dir=LINES_DIR)
print("Fine-tuned pipeline ready")

In [ ]:
# run pipeline with ablation on all eval sources
ft_results = {}

for source_name, data in all_eval_data.items():
    page_path = all_pages[source_name][0]
    page_id = f"{source_name}_ft"

    print(f"\n{source_name}: running fine-tuned pipeline...")
    stage_out = pipeline_ft.process_page_ablation(page_path, page_id=page_id)

    gt = data["gt_lines"]
    n = min(len(stage_out["stage4_final"]), len(gt))
    if n > 0:
        metrics = evaluate_lines(stage_out["stage4_final"][:n], gt[:n])
        print(f"  CER: {metrics['cer']:.4f}, WER: {metrics['wer']:.4f}, "
              f"NLS: {metrics['nls']:.4f}, Normalization errors: {metrics['normalization_errors']}")

    ft_results[source_name] = {
        "stage_outputs": stage_out,
        "ground_truth": gt,
    }

## 5. Zero-Shot Baseline Comparison

Run the same pipeline with a zero-shot VLM (no fine-tuning) to quantify the benefit of LoRA adaptation.

In [ ]:
# zero-shot Qwen (no adapter)
zeroshot_client = get_client("qwen")
pipeline_zs = OCRPipeline(zeroshot_client, few_shot_examples=few_shot, lines_dir=LINES_DIR)

zs_results = {}

for source_name, data in all_eval_data.items():
    page_path = all_pages[source_name][0]
    page_id = f"{source_name}_zs"

    print(f"\n{source_name}: running zero-shot pipeline...")
    stage_out = pipeline_zs.process_page_ablation(page_path, page_id=page_id)

    gt = data["gt_lines"]
    n = min(len(stage_out["stage4_final"]), len(gt))
    if n > 0:
        metrics = evaluate_lines(stage_out["stage4_final"][:n], gt[:n])
        print(f"  CER: {metrics['cer']:.4f}, WER: {metrics['wer']:.4f}, "
              f"NLS: {metrics['nls']:.4f}, Normalization errors: {metrics['normalization_errors']}")

    zs_results[source_name] = {
        "stage_outputs": stage_out,
        "ground_truth": gt,
    }

## 6. Per-Stage Ablation

Measure CER at each pipeline stage to show where VLM integration provides the most efficient gains.

In [ ]:
ablation_rows = []

for label, results_dict in [("Fine-tuned", ft_results), ("Zero-shot", zs_results)]:
    for source_name, data in results_dict.items():
        gt = data["ground_truth"]
        abl = evaluate_ablation(data["stage_outputs"], gt)
        for stage_label, metrics in abl.items():
            ablation_rows.append({
                "Model": label,
                "Source": source_name,
                "Stage": stage_label,
                "CER": metrics["cer"],
                "WER": metrics["wer"],
                "NLS": metrics["nls"],
                "Norm. Errors": metrics["normalization_errors"],
            })

ablation_df = pd.DataFrame(ablation_rows)
if not ablation_df.empty:
    print("Per-stage ablation:")
    print(ablation_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
if not ablation_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, model_label in zip(axes, ["Fine-tuned", "Zero-shot"]):
        sub = ablation_df[ablation_df["Model"] == model_label]
        avg = sub.groupby("Stage")["CER"].mean()
        avg.plot(kind="bar", ax=ax, color=["#e74c3c", "#f39c12", "#2ecc71"], rot=15)
        ax.set_ylabel("CER (lower is better)")
        ax.set_title(f"{model_label} -- CER by Pipeline Stage")
        ax.set_ylim(0, max(0.5, avg.max() * 1.3))

    plt.tight_layout()
    plt.show()

    norm_ft = ablation_df[ablation_df["Model"] == "Fine-tuned"]["Norm. Errors"].sum()
    norm_zs = ablation_df[ablation_df["Model"] == "Zero-shot"]["Norm. Errors"].sum()
    print(f"\nOrthographic normalization errors (historical -> modern substitutions):")
    print(f"  Fine-tuned: {norm_ft}")
    print(f"  Zero-shot:  {norm_zs}")

## 7. Discussion

**Evaluation metrics used:**
- **CER** — primary metric, character-level edit distance normalized by reference length
- **WER** — word-level, captures word boundary and spacing errors
- **NLS** — normalized Levenshtein similarity for cross-study comparability
- **Orthographic normalization errors** — counts instances where the model silently modernized historical spelling (e.g., "dixo" → "dijo"). Vesalainen et al. (2026) identified this as a critical concern for VLM-based OCR on historical text.

**Where VLM integration provides the most gains:**
- Stage 1 (layout): hybrid approach — projection profiles handle line geometry, VLM filters marginalia vs. main text
- Stage 2 (transcription): fine-tuned VLM with few-shot image+text examples from the same document
- Stage 3 (self-correction): the VLM re-reads the original image alongside its own output — this is where the largest CER improvement should appear, consistent with Greif et al. (2025)
- Stage 4 (context): text-only pass catches errors that require cross-line context

**Fine-tuning vs. zero-shot:** LoRA adaptation on even a small amount of ground truth data should substantially reduce CER, following the finding by Chung & Choi (2025) and Heidenreich et al. (2026, GutenOCR) that fine-tuning Qwen2.5-VL dramatically improves OCR performance on domain-specific documents.

**Limitations:**
- Ground truth is limited to initial portions of each PDF — evaluation on untranscribed pages would test generalization
- Line detection accuracy depends on page layout; densely annotated pages may need manual correction
- The orthographic preservation metric covers known normalization pairs but may miss less common historical forms